In [0]:
%run "/Workspace/Repos/shoyofromconcrete@gmail.com/MultiChannelDataPipeLine/config"

In [0]:
from pyspark.sql.types import*
from pyspark.sql.functions import*
from pyspark.sql.window import*

In [0]:
amazon_b=spark.table("multidatadumps.bronze.amazon_bronze")
pos_b=spark.table("multidatadumps.bronze.pos_bronze")
shopify_b=spark.table("multidatadumps.bronze.shopify_bronze")

display(amazon_b)
display(pos_b)
display(shopify_b)

In [0]:
amazon_cols=[]
pos_cols=[]
shopify_cols=[]

for a in amazon_b.columns:
    amazon_cols.append(a)
for p in pos_b.columns:
    pos_cols.append(p)
for s in shopify_b.columns:
    shopify_cols.append(s)
print(amazon_cols)
print(pos_cols)
print(shopify_cols)

In [0]:
COLUMN_MAPPINGS = {

    "amazon": {
        "id": "order_id",
        "buyer_name": "customer_name",
        "email": "email",
        "product_name": "product_name",
        "category": "category",
        "price": "price_per_unit",
        "quantity": "quantity",
        "discount": "discount",
        "tax": "tax",
        "payment_type": "payment_method",
        "order_status": "order_status",
        "purchase_timestamp": "order_date",
        "delivery_date": "delivery_date",
        "shipping_city": "shipping_city"
    },

    "pos": {
        "txn_id": "order_id",
        "customer_name": "customer_name",
        "item": "product_name",
        "category": "category",
        "price_per_unit": "price_per_unit",
        "qty": "quantity",
        "total_price": "total_amount",
        "discount_applied": "discount",
        "tax": "tax",
        "payment_mode": "payment_method",
        "date": "order_date",
        "store_location": "shipping_city"
    },

    "shopify": {
        "order_id": "order_id",
        "customer_name": "customer_name",
        "email": "email",
        "phone": "phone",
        "product": "product_name",
        "category": "category",
        "price_per_unit": "price_per_unit",
        "quantity": "quantity",
        "amount": "total_amount",
        "discount": "discount",
        "tax": "tax",
        "payment_method": "payment_method",
        "payment_status": "payment_status",
        "order_status": "order_status",
        "order_date": "order_date",
        "delivery_date": "delivery_date",
        "shipping_address": "shipping_city",
        "pincode": "pincode"
    }
}

In [0]:
standard_cols = [
    "order_id",
    "customer_name",
    "email",
    "phone",
    "product_name",
    "category",
    "price_per_unit",
    "quantity",
    "total_amount",
    "discount",
    "tax",
    "payment_method",
    "payment_status",
    "order_status",
    "order_date",
    "delivery_date",
    "shipping_city",
    "pincode",
    "source"
]

In [0]:
COMMON_COLS = [
    "source_file",
    "ingest_date",
    "start_ts",
    "end_ts",
    "is_current"
]

def standardize_df(df, source, standard_cols):
    
    mapping = COLUMN_MAPPINGS[source]

    # Rename mapped columns
    renamed_cols = [
        col(src).alias(dest)
        for src, dest in mapping.items()
        if src in df.columns
    ]

    # Keep common cols if they exist
    common_cols = [
        col(c) for c in COMMON_COLS if c in df.columns
    ]

    df_std = df.select(*renamed_cols, *common_cols)

    # Add missing standard columns
    for col_name in standard_cols:
        if col_name not in df_std.columns:
            df_std = df_std.withColumn(col_name, lit(None))

    # Add source column
    df_std = df_std.withColumn("source", lit(source))

    # Final column order (include common cols too)
    final_cols = standard_cols + COMMON_COLS
    df_std = df_std.select(*final_cols)

    return df_std

In [0]:
amazon_silver = standardize_df(amazon_b, "amazon", standard_cols)
pos_silver = standardize_df(pos_b, "pos", standard_cols)
shopify_silver = standardize_df(shopify_b, "shopify", standard_cols)

In [0]:
display(amazon_silver)
display(pos_silver)
display(shopify_silver)


Combine DataFrames now

In [0]:
final_df=amazon_silver.unionByName(pos_silver,allowMissingColumns=True).unionByName(shopify_silver,allowMissingColumns=True)
display(final_df)

Now Data Standardization , i mean we do this before or not idk your wish i find it simpler to combine data and then clean it

In [0]:
final_df.printSchema()

In [0]:
TARGET_SCHEMA = {
    "order_id": "string",
    "customer_name": "string",
    "email": "string",
    "phone": "string",
    "product_name": "string",
    "category": "string",
    "price_per_unit": "double",
    "quantity": "int",
    "total_amount": "double",
    "discount": "double",
    "tax": "double",
    "payment_method": "string",
    "payment_status": "string",
    "order_status": "string",
    "order_date": "date",
    "delivery_date": "date",
    "shipping_city": "string",
    "pincode": "string",
    "source": "string",
    "source_file": "string",
    "ingest_date": "date",
    "start_ts": "timestamp",
    "end_ts": "timestamp",
    "is_current": "int"
}

In [0]:
def parse_mixed_timestamp(col_name):
    
    c = col(col_name).cast("string")

    return (
        when(c.isNull(), None)

        .when(c.rlike(r"^\d{4}-\d{2}-\d{2}T"),
              to_timestamp(c))

        .when(c.rlike(r"^\d{4}-\d{2}-\d{2}$"),
              to_timestamp(c, "yyyy-MM-dd"))

        .when(c.rlike(r"^\d{2}/\d{2}/\d{2}$"),
              to_timestamp(c, "dd/MM/yy"))

        .when(c.rlike(r"^\d{10}$"),
              from_unixtime(c.cast("long")).cast("timestamp"))

        .when(c.rlike(r"^\d{13}$"),
              from_unixtime((c.cast("long") / 1000)).cast("timestamp"))

        .otherwise(None)
    )


def cast_columns(df, schema_dict):

    for col_name, dtype in schema_dict.items():

        if col_name in df.columns:

            current_type = dict(df.dtypes)[col_name]

            # -------------------------
            # TIMESTAMP
            # -------------------------
            if dtype == "timestamp":

                if current_type == "timestamp":
                    continue  # already correct → skip

                df = df.withColumn(col_name, parse_mixed_timestamp(col_name))

            # -------------------------
            #  DATE
            # -------------------------
            elif dtype == "date":

                if current_type == "date":
                    continue

                if current_type == "timestamp":
                    df = df.withColumn(col_name, to_date(col(col_name)))
                else:
                    df = df.withColumn(
                        col_name,
                        to_date(parse_mixed_timestamp(col_name))
                    )

            # -------------------------
            #  OTHER TYPES
            # -------------------------
            else:

                if current_type == dtype:
                    continue

                df = df.withColumn(col_name, col(col_name).cast(dtype))

    return df

In [0]:
final_df = cast_columns(final_df, TARGET_SCHEMA)
final_df=final_df.withColumn("discount",abs(col("discount").cast("double")))
display(final_df)
